# Green Taxi Data Stream Producer
This notebook demonstrates how to stream taxi trip records from a Parquet source into a Kafka topic for real-time processing.

### Step 1: Environment Setup
Import necessary libraries and enable autoreload.


In [1]:
%load_ext autoreload
%autoreload 2
import time
import pandas as pd


### Step 2: Project Path Setup
Add the project root to `sys.path` for module imports.


In [2]:
import sys
import os

# Add the project root (parent of notebooks/) to sys.path so we can import 'src' as a package
project_root = os.path.dirname(os.getcwd())
if project_root not in sys.path:
    sys.path.insert(0, project_root)


from src.models import GreenRide, green_ride_from_row, ride_serializer


url = "https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2025-10.parquet"

columns = [
    'lpep_pickup_datetime',
    'lpep_dropoff_datetime',
    'PULocationID',
    'DOLocationID',
    'passenger_count',
    'trip_distance',
    'tip_amount',
    'total_amount'
]

### Step 3: Import Custom Models
Import the `GreenRide` model and serialization logic.


### Step 4: Dataset Configuration
Define the data source URL and required columns.


### Step 5: Load Data
Read the taxi trip data from the Parquet source.


In [3]:
df = pd.read_parquet(url, columns=columns)

### Step 6: Data Preview
Verify the loaded data structure.


In [4]:
df.head()

,lpep_pickup_datetime,lpep_dropoff_datetime,PULocationID,DOLocationID,passenger_count,trip_distance,tip_amount,total_amount
0,2025-10-01 00:21:47,2025-10-01 00:24:37,247,69,1.0,0.70,1.70,10.00
1,2025-10-01 00:14:03,2025-10-01 00:24:14,66,25,1.0,1.61,2.78,16.68
2,2025-10-01 00:16:44,2025-10-01 00:16:47,244,244,1.0,0.00,2.20,13.20
3,2025-10-01 00:07:36,2025-10-01 00:32:14,95,170,1.0,10.37,11.31,67.85
4,2025-09-30 21:10:29,2025-09-30 21:22:30,82,138,1.0,4.07,6.82,34.12


### Step 7: Kafka Initialization
Initialize the Kafka producer.


In [5]:
from kafka import KafkaProducer

topic_name = 'green-trips'
producer = KafkaProducer(
    bootstrap_servers=['localhost:9092'],
    value_serializer=ride_serializer
)

### Step 8: Data Production
Stream the data to the Kafka topic.


In [7]:
import time

t0 = time.time()
count = 0

for _, row in df.iterrows():
    ride = green_ride_from_row(row)
    producer.send(topic_name, value=ride)
    count += 1
    if count % 100 == 0:
        print(f"Sent {count} messages...")
    time.sleep(0.01)

producer.flush()

t1 = time.time()
print(f'Done! Sent {count} total messages.')
print(f'took {(t1 - t0):.2f} seconds')

Sent 100 messages...
Sent 200 messages...
Sent 300 messages...
Sent 400 messages...
Sent 500 messages...
Sent 600 messages...
Sent 700 messages...
Sent 800 messages...
Sent 900 messages...
Sent 1000 messages...
Sent 1100 messages...
Sent 1200 messages...
Sent 1300 messages...
Sent 1400 messages...
Sent 1500 messages...
Sent 1600 messages...
Sent 1700 messages...
Sent 1800 messages...
Sent 1900 messages...
Sent 2000 messages...
Sent 2100 messages...
Sent 2200 messages...
Sent 2300 messages...
Sent 2400 messages...
Sent 2500 messages...
Sent 2600 messages...
Sent 2700 messages...
Sent 2800 messages...
Sent 2900 messages...
Sent 3000 messages...
Sent 3100 messages...
Sent 3200 messages...
Sent 3300 messages...
Sent 3400 messages...
Sent 3500 messages...
Sent 3600 messages...
Sent 3700 messages...
Sent 3800 messages...
Sent 3900 messages...
Sent 4000 messages...
Sent 4100 messages...
Sent 4200 messages...
Sent 4300 messages...
Sent 4400 messages...
Sent 4500 messages...
Sent 4600 messages.

### Step 9: Resource Cleanup

In [ ]:
producer.close()
print("Producer closed.")